# Support Vector Machine (SVM)

In this notebook, we will explore the **Support Vector Machine** — one of the most powerful and elegant supervised learning algorithms for classification (and regression). We will build a simplified linear SVM from scratch using pure Python.

## 1. Introduction — What Is SVM?

A **Support Vector Machine** is a supervised learning algorithm that finds the **optimal decision boundary** (called a *hyperplane*) that separates data points of different classes.

The key insight that distinguishes SVM from other classifiers:

> Among **all** hyperplanes that correctly separate two classes, SVM chooses the one that **maximises the margin** — the distance between the decision boundary and the nearest data points from each class.

### Why maximise the margin?

A larger margin means the classifier is more robust to small perturbations in the data and, in practice, generalises better to unseen examples. This idea is formalised in **statistical learning theory** (Vapnik–Chervonenkis theory).

### Maximum Margin Classifier

Imagine you have two groups of points on a 2-D plane that can be separated by a straight line. Many lines can separate them, but the **maximum margin classifier** picks the line that sits exactly in the middle of the widest *street* you can draw between the two classes.

The data points that lie on the edges of this street are called **support vectors** — they are the only points that matter for defining the boundary.

## 2. Geometric Intuition

### Decision Boundary (Hyperplane)

In *n* dimensions, the decision boundary is an (n−1)-dimensional hyperplane defined by:

$$\mathbf{w} \cdot \mathbf{x} + b = 0$$

where **w** is the weight (normal) vector and *b* is the bias.

- Points on one side satisfy $\mathbf{w} \cdot \mathbf{x} + b > 0$ → class +1  
- Points on the other side satisfy $\mathbf{w} \cdot \mathbf{x} + b < 0$ → class −1

### Margin

The **margin** is the perpendicular distance between the two parallel *gutter* hyperplanes:

$$\mathbf{w} \cdot \mathbf{x} + b = +1 \quad \text{and} \quad \mathbf{w} \cdot \mathbf{x} + b = -1$$

The width of this margin is:

$$\text{margin} = \frac{2}{\|\mathbf{w}\|}$$

So to **maximise** the margin, we need to **minimise** $\|\mathbf{w}\|$.

### Support Vectors

Support vectors are the training examples that sit exactly on the margin boundaries ($\mathbf{w} \cdot \mathbf{x} + b = \pm 1$). These are the hardest-to-classify points and they alone determine the position and orientation of the decision boundary. If you removed any non-support-vector point from the dataset, the boundary would not change.

## 3. Hard Margin SVM

When the data is **perfectly linearly separable**, we can use the *hard margin* formulation.

### Optimisation Problem

We want to find **w** and *b* that:

$$\min_{\mathbf{w}, b} \; \frac{1}{2} \|\mathbf{w}\|^2$$

subject to the constraint that every training point is correctly classified with at least unit functional margin:

$$y_i \, (\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1, \quad \forall \; i = 1, \dots, N$$

where $y_i \in \{-1, +1\}$.

### Interpretation

- Minimising $\frac{1}{2}\|\mathbf{w}\|^2$ is equivalent to maximising the margin $\frac{2}{\|\mathbf{w}\|}$.
- The constraint ensures no data point falls inside the margin.
- This is a **convex quadratic programming** problem — it has a unique global solution.

### Limitation

Hard margin SVM **fails** if the data is not perfectly linearly separable — even a single outlier can make it infeasible.

## 4. Soft Margin SVM

Real-world data is rarely perfectly separable. The **soft margin** formulation introduces **slack variables** $\xi_i \geq 0$ that allow some points to violate the margin (or even be misclassified).

### Optimisation Problem

$$\min_{\mathbf{w}, b, \xi} \; \frac{1}{2} \|\mathbf{w}\|^2 + C \sum_{i=1}^{N} \xi_i$$

subject to:

$$y_i \, (\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

### The C Parameter

**C** controls the trade-off between two competing goals:

| C value | Effect |
|---------|--------|
| Large C | Penalises misclassifications heavily → narrower margin, fewer violations (risk of overfitting) |
| Small C | Allows more margin violations → wider margin, more tolerance (risk of underfitting) |

- $\xi_i = 0$: the point is correctly classified and outside the margin.  
- $0 < \xi_i < 1$: correctly classified but inside the margin.  
- $\xi_i \geq 1$: the point is misclassified.

## 5. The Kernel Trick

### Motivation

Many real datasets are **not** linearly separable in their original feature space. For example, imagine two concentric circles of data — no straight line can separate them.

The idea: **map the data into a higher-dimensional space** where it *becomes* linearly separable, then find the optimal hyperplane there.

### How It Works

Instead of computing the mapping $\phi(\mathbf{x})$ explicitly (which can be very expensive), we use a **kernel function** $K(\mathbf{x}_i, \mathbf{x}_j)$ that computes the dot product in the higher-dimensional space directly:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i) \cdot \phi(\mathbf{x}_j)$$

This is possible because the SVM optimisation (in its dual form) only depends on dot products between data points.

### Common Kernels

| Kernel | Formula | Use Case |
|--------|---------|----------|
| **Linear** | $K(\mathbf{x}_i, \mathbf{x}_j) = \mathbf{x}_i \cdot \mathbf{x}_j$ | Linearly separable data, high-dimensional sparse data (e.g. text) |
| **Polynomial** | $K(\mathbf{x}_i, \mathbf{x}_j) = (\gamma \, \mathbf{x}_i \cdot \mathbf{x}_j + r)^d$ | Interaction features, image processing |
| **RBF (Gaussian)** | $K(\mathbf{x}_i, \mathbf{x}_j) = \exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$ | Default choice, maps into infinite-dimensional space |

The **RBF kernel** is the most popular because it can model arbitrarily complex boundaries (given enough support vectors), and it has only one hyperparameter ($\gamma$).

## 6. Simplified Linear SVM Implementation (SGD)

We will implement a **linear SVM** using **stochastic gradient descent** on the **hinge loss** with L2 regularisation.

### Loss Function

For each sample $(\mathbf{x}_i, y_i)$:

$$L_i = \frac{\lambda}{2} \|\mathbf{w}\|^2 + \max(0, \; 1 - y_i \, (\mathbf{w} \cdot \mathbf{x}_i + b))$$

where $\lambda = \frac{1}{C}$ is the regularisation strength.

### Gradient Update Rules

If the point is **correctly classified** with sufficient margin ($y_i (\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1$):

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \, \lambda \, \mathbf{w}$$

If the point **violates the margin** ($y_i (\mathbf{w} \cdot \mathbf{x}_i + b) < 1$):

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \, (\lambda \, \mathbf{w} - y_i \, \mathbf{x}_i)$$
$$b \leftarrow b + \eta \, y_i$$

In [ ]:
import random
import math


def dot(a, b):
    """Dot product of two vectors."""
    return sum(ai * bi for ai, bi in zip(a, b))


def vector_scale(v, s):
    """Multiply every element of v by scalar s."""
    return [vi * s for vi in v]


def vector_add(a, b):
    return [ai + bi for ai, bi in zip(a, b)]


def vector_sub(a, b):
    return [ai - bi for ai, bi in zip(a, b)]


def vector_norm(v):
    return math.sqrt(sum(vi * vi for vi in v))

In [ ]:
def svm_sgd_train(X, y, learning_rate=0.001, lambda_param=0.01, n_epochs=1000):
    """
    Train a linear SVM using stochastic gradient descent on hinge loss.

    Parameters
    ----------
    X : list of list of float  – training features
    y : list of int            – labels in {-1, +1}
    learning_rate : float      – step size for SGD
    lambda_param : float       – regularisation strength (1/C)
    n_epochs : int             – passes over the entire dataset

    Returns
    -------
    w : list of float – weight vector
    b : float         – bias term
    """
    n_samples = len(X)
    n_features = len(X[0])

    w = [0.0] * n_features
    b = 0.0

    for epoch in range(n_epochs):
        indices = list(range(n_samples))
        random.shuffle(indices)

        for i in indices:
            condition = y[i] * (dot(w, X[i]) + b)

            if condition >= 1:
                w = vector_sub(w, vector_scale(w, learning_rate * lambda_param))
            else:
                w = vector_sub(
                    w,
                    vector_sub(
                        vector_scale(w, learning_rate * lambda_param),
                        vector_scale(X[i], learning_rate * y[i]),
                    ),
                )
                b += learning_rate * y[i]

    return w, b

In [ ]:
def svm_predict_one(x, w, b):
    """Predict class label for a single sample."""
    return 1 if dot(w, x) + b >= 0 else -1


def svm_predict(X, w, b):
    """Predict class labels for a list of samples."""
    return [svm_predict_one(x, w, b) for x in X]

## 7. Worked Example

We will create a small, linearly separable 2-D dataset and train our SVM on it.

In [ ]:
random.seed(42)

def make_cluster(cx, cy, n, spread=0.8):
    return [[cx + random.uniform(-spread, spread),
             cy + random.uniform(-spread, spread)] for _ in range(n)]

pos_points = make_cluster(3, 3, 20)
neg_points = make_cluster(0, 0, 20)

X_train = pos_points + neg_points
y_train = [1] * 20 + [-1] * 20

print(f'Dataset size: {len(X_train)} samples, {len(X_train[0])} features')
print(f'First 3 positive: {pos_points[:3]}')
print(f'First 3 negative: {neg_points[:3]}')

In [ ]:
w, b = svm_sgd_train(
    X_train, y_train,
    learning_rate=0.001,
    lambda_param=0.01,
    n_epochs=1000,
)

print(f'Learned weights: w = [{w[0]:.4f}, {w[1]:.4f}]')
print(f'Learned bias:    b = {b:.4f}')
print(f'Weight norm:     ||w|| = {vector_norm(w):.4f}')
print(f'Approx margin:   {2 / vector_norm(w):.4f}')

In [ ]:
predictions = svm_predict(X_train, w, b)
correct = sum(1 for p, t in zip(predictions, y_train) if p == t)
accuracy = correct / len(y_train) * 100

print(f'Training accuracy: {correct}/{len(y_train)} = {accuracy:.1f}%')

In [ ]:
X_test = [
    [3.5, 3.5],
    [0.5, 0.2],
    [2.0, 2.5],
    [-0.5, 0.5],
    [4.0, 3.0],
]
y_test = [1, -1, 1, -1, 1]

test_preds = svm_predict(X_test, w, b)

print('Test predictions:')
for xi, yi, pi in zip(X_test, y_test, test_preds):
    status = 'correct' if yi == pi else 'WRONG'
    print(f'  x={xi}, true={yi:+d}, pred={pi:+d}  [{status}]')

In [ ]:
print('Decision function values (distance from boundary):')
for xi in X_test:
    score = dot(w, xi) + b
    print(f'  x={xi}  =>  w*x + b = {score:.4f}')

### Understanding the Output

- The **weight vector** `w` points perpendicular to the decision boundary.  
- The **bias** `b` shifts the boundary away from the origin.  
- The **decision function** value `w*x + b` tells us how far a point is from the boundary:  
  - Large positive → confidently class +1  
  - Large negative → confidently class −1  
  - Near zero → close to the decision boundary

## 8. Hyperparameters

### C (Regularisation Parameter)

In our implementation, `lambda_param = 1/C`.

| C | lambda | Behaviour |
|---|--------|-----------|
| 100 | 0.01 | Strict: few margin violations allowed |
| 1 | 1.0 | Balanced |
| 0.01 | 100 | Relaxed: wide margin, many violations tolerated |

**Tuning tip**: Use cross-validation to search over a logarithmic grid, e.g. C in {0.001, 0.01, 0.1, 1, 10, 100}.

### Kernel Choice

- **Linear**: Start here. Works well for high-dimensional data or when n_features >> n_samples (e.g. text classification).  
- **RBF**: Default for non-linear problems. Most flexible.  
- **Polynomial**: Useful when you suspect interactions up to degree *d*.

### Gamma (for RBF and Polynomial kernels)

Controls the *reach* of each training example:

| Gamma | Effect |
|-------|--------|
| Small | Each point has far-reaching influence → smoother boundary (may underfit) |
| Large | Each point only influences nearby points → complex, wiggly boundary (may overfit) |

A common default is $\gamma = \frac{1}{n_{\text{features}}}$.

In [ ]:
print('Effect of regularisation strength (lambda_param):')
print('=' * 55)

for lam in [0.001, 0.01, 0.1, 1.0]:
    w_exp, b_exp = svm_sgd_train(
        X_train, y_train,
        learning_rate=0.001,
        lambda_param=lam,
        n_epochs=1000,
    )
    preds = svm_predict(X_train, w_exp, b_exp)
    acc = sum(1 for p, t in zip(preds, y_train) if p == t) / len(y_train)
    margin = 2 / vector_norm(w_exp) if vector_norm(w_exp) > 0 else float('inf')
    print(f'  lambda={lam:<6}  C={1/lam:<8.1f}  ||w||={vector_norm(w_exp):.4f}  '
          f'margin={margin:.4f}  accuracy={acc:.2%}')

## 9. Summary

### When to Use SVM

- **Binary classification** with clear (or near-clear) margin of separation.  
- **High-dimensional** data (e.g. text/NLP) where linear SVM excels.  
- **Small to medium** datasets — SVM scales as roughly $O(n^2)$ to $O(n^3)$ with standard solvers.

### Strengths

- Effective in high-dimensional spaces (even when n_features > n_samples).  
- Memory-efficient: only support vectors are stored for prediction.  
- Versatile through different kernel functions.  
- Strong theoretical guarantees on generalisation (margin theory).

### Weaknesses

- Slow training on very large datasets.  
- Sensitive to feature scaling — **always normalise/standardise first**.  
- Does not directly provide probability estimates (requires additional calibration).  
- Choice of kernel and hyperparameters (C, gamma) can significantly affect performance.

### Key Takeaways

1. SVM finds the **maximum margin** hyperplane.  
2. **Soft margin** (with parameter C) handles non-separable data.  
3. The **kernel trick** enables non-linear decision boundaries without explicit feature mapping.  
4. In practice, linear SVM with SGD is efficient and often a strong baseline.  
5. Always **scale your features** and **tune C** (and gamma for RBF) via cross-validation.